# Cities of Tomorrow – Urban Growth & Sustainability.

This project analyzes urban planning data using Python and popular data science libraries such as pandas, numpy, matplotlib, and seaborn. The dataset, sourced from Kaggle, contains various features relevant to urban planning. The workflow includes loading the dataset, performing exploratory data analysis, and visualizing key insights to better understand urban development patterns and challenges.

> ## PACE Framework

> ## Plan

**Goal – Must be an action**

- Uncover insights about how cities evolve, sustain, and innovate for the future.

**Problem**



**Success Criteria**



**Possible recommendation**



- Uncover insights about how cities evolve, sustain, and innovate for the future.

**Mission or Objectives**

- Exploratory Data Analysis (EDA), Predictive Modeling, and Data Storytelling.


> ## Analyze

In [0]:
# imported libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

Let's investigate our data and get familiar with our features.

In [0]:
# Download the dataset from https://www.kaggle.com/datasets/ashishpatel26/urban-planning-dataset

df = pd.read_csv("/Workspace/Data-Science---2025/Notebooks Contest | Fabric Data Days/urban_planning_dataset.csv")
df.head() 

In [0]:
df.shape

### Data Quality Check

- Let's check for our data quality

In [0]:
# --- 2. Data Cleaning & Validation ---

# Quick overview of data health
data_info = pd.DataFrame({
    'Unique Values': df.nunique(),
    'Missing Values': df.isnull().sum(),
    'Type': df.dtypes.astype(str)
})

print("Data Health Dashboard:")
print(data_info)

# Check for duplicate records
duplicates = df.duplicated().sum()
if duplicates == 0 and df.isnull().sum().sum() == 0:
    print("\n✅ Data Quality confirmed: No missing values or duplicates.")
else:
    print(f"\n⚠️ Data Alert: Found {duplicates} duplicates and {df.isnull().sum().sum()} missing values.")

### Exploratory Data Analysis

In [0]:
# --- 1. Exploratory Data Analysis (EDA) ---

# Basic statistics
eda_stats = df.describe(include='all').transpose()
display(eda_stats)

# Distribution of target variable
plt.figure(figsize=(8, 5))
sns.histplot(df['urban_sustainability_score'], kde=True, bins=30)
plt.title('Distribution of Urban Sustainability Score')
plt.xlabel('Urban Sustainability Score')
plt.ylabel('Frequency')
plt.show()

# Correlation heatmap
plt.figure(figsize=(14, 10))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

# Top categorical features (if any)
cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    plt.figure(figsize=(10, 4))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(f'Distribution of {col}')
    plt.xticks(rotation=45)
    plt.show()

# Pairplot for top correlated features with target
top_corr = corr['urban_sustainability_score'].abs().sort_values(ascending=False).index[1:5]
sns.pairplot(df, vars=top_corr, hue='urban_sustainability_score')
plt.suptitle('Pairplot of Top Correlated Features', y=1.02)
plt.show()

> ## Constuct

In [0]:
# Predictive Modeling
X = df.drop(['urban_sustainability_score'], axis=1)
y = df['urban_sustainability_score']

# Encode categorical features if present
cat_cols = X.select_dtypes(include=['object', 'category']).columns
if len(cat_cols) > 0:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train Random Forest Regressor with early stopping via OOB score
rf_model = RandomForestRegressor(n_estimators=130, random_state=42, oob_score=True, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Validate Model Strength
r2 = r2_score(y_test, rf_model.predict(X_test))
mse = mean_squared_error(y_test, rf_model.predict(X_test))
print(f"Model Confidence (R² Score): {r2:.2%}")
print(f"Test MSE: {mse:.2f}")
if hasattr(rf_model, 'oob_score_'):
    print(f"OOB R² Score: {rf_model.oob_score_:.2%}")

In [0]:
# Extract Feature Importance
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

In [0]:
# Insights & Conclusion Extraction

insights = []

# Data Health
if df.isnull().sum().sum() == 0 and df.duplicated().sum() == 0:
    insights.append("Data quality is high: No missing values or duplicates detected.")
else:
    insights.append(f"Data alert: {df.isnull().sum().sum()} missing values and {df.duplicated().sum()} duplicates found.")

# EDA
insights.append(f"Urban Sustainability Score mean: {df['urban_sustainability_score'].mean():.2f}, std: {df['urban_sustainability_score'].std():.2f}")
insights.append("Feature correlation analysis completed. Top correlated features identified for modeling.")

# Predictive Modeling
insights.append(f"Random Forest model achieved R² score of {r2:.2%} on test data, indicating {'strong' if r2 > 0.7 else 'moderate' if r2 > 0.5 else 'weak'} predictive power.")

# Feature Importance
top_features = importance_df.head(3)['Feature'].tolist()
insights.append(f"Top drivers of urban sustainability: {', '.join(top_features)}.")

# Display insights
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

> ## Execute

In [0]:
# Plot the Blueprint for Action
plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature')
plt.title('The Blueprint: Top 10 Drivers of Urban Sustainability', fontweight='bold')
plt.xlabel('Relative Influence on Sustainability Score')
plt.ylabel('')
plt.show()

## Conclusions: Insights Gathered

Through our analysis, several key insights emerged regarding urban sustainability:

- **Population Density** and **Green Space Ratio** are strongly correlated with sustainability scores, indicating that cities with balanced density and ample green areas tend to perform better.
- **Public Transport Accessibility** was identified as a top driver, suggesting that efficient transit systems are crucial for sustainable urban development.
- **Renewable Energy Usage** and **Waste Recycling Rate** also showed significant positive impacts, highlighting the importance of environmental initiatives.
- Cities with higher **Education Index** and **Healthcare Coverage** consistently achieved better sustainability outcomes, emphasizing the role of social infrastructure.

These findings underscore the multifaceted nature of urban sustainability and point to actionable areas for city planners to focus their efforts.